# LangChain Runnables — Building a Content Script Generator

This notebook teaches five core LangChain **Runnable** primitives by using them to build a real
pipeline: a tool that finds trending topics, filters out political ones by vote, researches the
survivors, and writes a short video script for each one.

**Runnables covered**
1. `RunnablePassthrough` — forward input unchanged so a later step can still see it
2. `RunnableParallel` — run several branches concurrently, merge into one dict
3. `RunnableLambda` — wrap any plain Python function as a Runnable
4. `RunnableBranch` — route to different sub-chains based on a condition
5. `.with_fallbacks(...)` (`RunnableWithFallbacks`) — automatic fallback if a step raises

**The pipeline (3 chains)**
- **Chain 1 — Topic Discovery:** Tavily-search trending news, LLM-vote on whether each headline
  is political, drop political ones, re-search to backfill until we have 3 clean topics.
- **Chain 2 — Research:** for each topic, deep-search + summarize, save raw findings to a `.txt` file.
- **Chain 3 — Script Writer:** load the saved research and generate a short-form video script per topic.

Model: `gpt-4o-mini` · Search: Tavily

## 0. Setup

In [17]:
#!pip install -q langchain langchain-openai langchain-community tavily-python

In [18]:
import os
import getpass
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda, RunnableBranch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
search_tool = TavilySearchResults(max_results=5)

# Tavily results don't always include a clean 'title' field depending on API version,
# so we build a small helper that always returns something usable as a headline.
def extract_title(result: dict) -> str:
    return result.get("title") or (result.get("content", "")[:80] + "...")

## Part 1 — The Five Building Blocks
A quick, isolated demo of each Runnable before we wire them into the real pipeline.

### 1. `RunnablePassthrough`
Forwards its input unchanged. On its own it's trivial — its real value shows up **inside**
a `RunnableParallel`, where it lets a later step see the *original* input alongside a
transformed version of it (e.g. keep the topic string around while a search runs on it).

In [19]:
demo_passthrough = RunnableParallel(
    original=RunnablePassthrough(),
    upper=RunnableLambda(lambda x: x.upper()),
)
demo_passthrough.invoke("trending topic")

{'original': 'trending topic', 'upper': 'TRENDING TOPIC'}

### 2. `RunnableParallel`
Runs multiple branches on the *same* input at once and merges their outputs into a dict.
We'll use this both to cast several LLM "votes" simultaneously, and to run a web search
and a passthrough side by side.

In [20]:
demo_parallel = RunnableParallel(
    length=RunnableLambda(lambda x: len(x)),
    words=RunnableLambda(lambda x: x.split()),
)
demo_parallel.invoke("Runnables make LCEL composable")

{'length': 30, 'words': ['Runnables', 'make', 'LCEL', 'composable']}

### 3. `RunnableLambda`
Wraps any plain Python function so it can be composed with `|`. This is the glue that lets
ordinary Python (parsing, formatting, file I/O) sit inside an LCEL chain.

In [21]:
def parse_titles(raw_results):
    return [extract_title(r) for r in raw_results]

parse_titles_runnable = RunnableLambda(parse_titles)
parse_titles_runnable.invoke([{"title": "Example headline"}, {"content": "No title here, just body text"}])

['Example headline', 'No title here, just body text...']

### 4. `RunnableBranch`
Conditional routing: give it `(condition, runnable)` pairs plus a default. The first condition
that returns `True` wins; otherwise the default runs. We'll use this to route "political" vs
"keep" topics to different handling.

In [22]:
sign_branch = RunnableBranch(
    (lambda x: x > 0, RunnableLambda(lambda x: "positive")),
    (lambda x: x < 0, RunnableLambda(lambda x: "negative")),
    RunnableLambda(lambda x: "zero"),  # default
)
print(sign_branch.invoke(5), sign_branch.invoke(-3), sign_branch.invoke(0))

positive negative zero


### 5. `.with_fallbacks(...)` — `RunnableWithFallbacks`
Wraps a runnable with one or more backups. If the primary raises an exception, the next
fallback is tried automatically. We'll use this around the Tavily search call and around the
script-writing LLM call so a single flaky request doesn't kill the whole pipeline.

In [23]:
def flaky(x):
    raise ValueError("simulated API failure")

primary = RunnableLambda(flaky)
backup = RunnableLambda(lambda x: f"[fallback result for {x}]")

robust_chain = primary.with_fallbacks([backup])
robust_chain.invoke("topic")

'[fallback result for topic]'

## Part 2 — Chain 1: Trending Topic Discovery
Search for trending headlines, cast a simple 3-way LLM vote on whether each one is political,
drop the political ones, and re-search to backfill until we have exactly 3 clean topics.

In [24]:
def run_trending_search(_input=None):
    return search_tool.invoke("top trending news topics today in sports")

def dedupe_titles(results):
    seen, titles = set(), []
    for r in results:
        t = extract_title(r).strip()
        if t and t not in seen:
            seen.add(t)
            titles.append(t)
    return titles

# RunnableLambda -> RunnableLambda composition via '|'
extract_topics = RunnableLambda(run_trending_search) | RunnableLambda(dedupe_titles)

**The vote.** For each candidate headline we ask the model 3 times whether it's political
and take the majority. `RunnableParallel` fires the 3 votes concurrently, and
`RunnablePassthrough` keeps the original topic string alive next to the votes.

In [25]:
vote_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with exactly one word: POLITICAL or NOT_POLITICAL. "
               "Judge whether the headline below is primarily about politics, "
               "government, elections, or policy."),
    ("human", "{topic}"),
])

single_vote = (
    RunnableLambda(lambda topic: {"topic": topic})
    | vote_prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    | StrOutputParser()
)

vote_panel = RunnableParallel(
    v1=single_vote,
    v2=single_vote,
    v3=single_vote,
    topic=RunnablePassthrough(),  # <- original topic string survives alongside the 3 votes
)

def tally_votes(votes):
    ballots = [votes["v1"], votes["v2"], votes["v3"]]
    political_votes = sum(1 for b in ballots if "NOT" not in b.upper())
    return {"topic": votes["topic"], "is_political": political_votes >= 2}

vote_and_tally = vote_panel | RunnableLambda(tally_votes)

In [26]:
keep_branch = RunnableLambda(lambda x: {"topic": x["topic"], "status": "kept"})
reject_branch = RunnableLambda(lambda x: {"topic": x["topic"], "status": "rejected_political"})

filter_branch = RunnableBranch(
    (lambda x: x["is_political"], reject_branch),
    keep_branch,  # default
)

topic_filter_chain = vote_and_tally | filter_branch
topic_filter_chain.invoke("Senate passes new spending bill after late-night vote")

{'topic': 'Senate passes new spending bill after late-night vote',
 'status': 'rejected_political'}

**Orchestration loop.** LCEL chains are single passes; the "search again to backfill" logic
is a small, explicit Python loop that calls the chain repeatedly. This is normal practice —
LCEL composes one step of work, and ordinary Python drives multi-round control flow around it.

In [27]:
def find_non_political_topics(target_count=3, max_rounds=5):
    accepted, seen_titles = [], set()
    rounds = 0
    while len(accepted) < target_count and rounds < max_rounds:
        rounds += 1
        print(f"--- search round {rounds} ---")
        candidates = extract_topics.invoke(None)
        needed = target_count - len(accepted)
        new_candidates = [t for t in candidates if t not in seen_titles][: needed + 2]
        for title in new_candidates:
            seen_titles.add(title)
            result = topic_filter_chain.invoke(title)
            print(f"  [{result['status']}] {result['topic']}")
            if result["status"] == "kept":
                accepted.append(result["topic"])
            if len(accepted) == target_count:
                break
    return accepted[:target_count]

trending_topics = find_non_political_topics()
print("\nFinal topics:", trending_topics)

--- search round 1 ---
  [kept] Trending | Trending News, Scores, Highlights, Stats, Standings, and Rumors | Bleacher Report
  [kept] Sky Sports - Sports News, Transfers, Scores | Watch Live Sport
  [kept] Sports - The New York Times

Final topics: ['Trending | Trending News, Scores, Highlights, Stats, Standings, and Rumors | Bleacher Report', 'Sky Sports - Sports News, Transfers, Scores | Watch Live Sport', 'Sports - The New York Times']


## Part 3 — Chain 2: Research + Save Raw Text
For each surviving topic: deep-search for details, summarize into bullet points, and write
both the summary and raw search hits to a `.txt` file. The search step is wrapped with
`.with_fallbacks(...)` so a failed/rate-limited call degrades gracefully instead of crashing.

In [28]:
def deep_search(topic):
    return search_tool.invoke(f"{topic} latest news details background")

def deep_search_safe(topic):
    # Fallback: a smaller, separate search call in case the primary one fails
    backup_tool = TavilySearchResults(max_results=2)
    return backup_tool.invoke(topic)

deep_search_runnable = RunnableLambda(deep_search).with_fallbacks([RunnableLambda(deep_search_safe)])

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant. Summarize the search results into clear, "
               "factual bullet points a scriptwriter can use. No fluff, no speculation."),
    ("human", "Topic: {topic}\n\nSearch results:\n{results}"),
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

In [29]:
# Stage A: fetch topic (passthrough) and search results (fallback-protected) side by side
gather_stage = RunnableParallel(
    topic=RunnablePassthrough(),
    search_results=deep_search_runnable,
)

def build_summary_input(x):
    return {"topic": x["topic"], "results": x["search_results"]}

# Stage B: from the gathered dict, produce topic / summary / raw_results in parallel
research_chain = gather_stage | RunnableParallel(
    topic=RunnableLambda(lambda x: x["topic"]),
    summary=RunnableLambda(build_summary_input) | summarize_chain,
    raw_results=RunnableLambda(lambda x: x["search_results"]),
)

In [30]:
RAW_DIR = Path("raw_research")
RAW_DIR.mkdir(exist_ok=True)

def safe_filename(topic: str) -> str:
    return "".join(c if c.isalnum() or c in " _-" else "_" for c in topic)[:60].strip()

def save_raw_text(record):
    path = RAW_DIR / f"{safe_filename(record['topic'])}.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"TOPIC: {record['topic']}\n\n")
        f.write("=== SUMMARY ===\n")
        f.write(record["summary"] + "\n\n")
        f.write("=== RAW SEARCH RESULTS ===\n")
        for r in record["raw_results"]:
            f.write(f"- {extract_title(r)}\n")
            f.write(f"  {r.get('url', '')}\n")
            f.write(f"  {r.get('content', '')[:500]}\n\n")
    record["raw_path"] = str(path)
    return record

full_research_chain = research_chain | RunnableLambda(save_raw_text)

research_records = [full_research_chain.invoke(t) for t in trending_topics]
for rec in research_records:
    print(f"Saved research for '{rec['topic']}' -> {rec['raw_path']}")

Saved research for 'Trending | Trending News, Scores, Highlights, Stats, Standings, and Rumors | Bleacher Report' -> raw_research\Trending _ Trending News_ Scores_ Highlights_ Stats_ Standin.txt
Saved research for 'Sky Sports - Sports News, Transfers, Scores | Watch Live Sport' -> raw_research\Sky Sports - Sports News_ Transfers_ Scores _ Watch Live Spo.txt
Saved research for 'Sports - The New York Times' -> raw_research\Sports - The New York Times.txt


## Part 4 — Chain 3: Script Writer
Load the saved research back off disk and turn it into a short-form video script per topic.
The writing LLM call itself is wrapped with `.with_fallbacks(...)` — if the primary call errors
out (rate limit, timeout, etc.) a backup configuration takes over automatically.

In [31]:
def load_raw_text(record):
    with open(record["raw_path"], "r", encoding="utf-8") as f:
        text = f.read()
    return {"topic": record["topic"], "research": text}

load_research = RunnableLambda(load_raw_text)

script_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a scriptwriter for short-form video content (60-90 seconds). "
               "Write an engaging, factual script with a hook, 2-3 key points, and a "
               "closing line. Use only the research provided; do not invent facts."),
    ("human", "Topic: {topic}\n\nResearch notes:\n{research}"),
])

primary_writer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
backup_writer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)  # simpler, steadier retry config
script_llm = primary_writer_llm.with_fallbacks([backup_writer_llm])

script_writer_chain = load_research | script_prompt | script_llm | StrOutputParser()

In [32]:
SCRIPT_DIR = Path("scripts")
SCRIPT_DIR.mkdir(exist_ok=True)

for rec in research_records:
    script_text = script_writer_chain.invoke(rec)
    script_path = SCRIPT_DIR / f"{safe_filename(rec['topic'])}_script.txt"
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(f"SCRIPT: {rec['topic']}\n\n{script_text}")
    print(f"\n{'='*60}\nSCRIPT for: {rec['topic']}\n{'='*60}\n{script_text}\n")


SCRIPT for: Trending | Trending News, Scores, Highlights, Stats, Standings, and Rumors | Bleacher Report
[Opening shot: Fast-paced montage of NBA highlights, trade rumors, and fan reactions]

**Narrator:** "What’s buzzing in the world of sports? Let’s dive into the latest trending news from Bleacher Report!”

[Cut to a Lakers logo with highlights of Jonathan Kuminga]

**Narrator:** “First up, the Los Angeles Lakers are reportedly eyeing Jonathan Kuminga for a trade! Will this young talent be donning the purple and gold soon?”

[Transition to a graphic of top NBA free agents]

**Narrator:** “And speaking of trades, a fresh list of the top remaining NBA free agents has dropped. Teams are gearing up for the upcoming season, and every move could be pivotal!”

[Quick cut to footage of Novak Djokovic celebrating]

**Narrator:** “Switching gears to tennis, Novak Djokovic has made history! He just won the longest quarterfinal match at Wimbledon, an epic battle that showcased his incredible en

## Recap — Where Each Runnable Was Used

| Runnable | Where it showed up | Why |
|---|---|---|
| `RunnablePassthrough` | `vote_panel`, `gather_stage` | keep the original topic string alive next to derived branches |
| `RunnableParallel` | vote casting, search+passthrough, topic/summary/raw split | run independent work concurrently, merge into one dict |
| `RunnableLambda` | search wrappers, parsers, file I/O, tally logic | bring plain Python into an LCEL chain |
| `RunnableBranch` | `filter_branch` | route "political" vs "keep" topics differently |
| `.with_fallbacks(...)` | Tavily deep search, script-writing LLM call | degrade gracefully instead of crashing on a flaky call |

**Output on disk after running this notebook:**
- `raw_research/*.txt` — one raw research file per approved topic
- `scripts/*.txt` — one generated script per approved topic